<a href="https://colab.research.google.com/github/Grashch/Computer-Vision/blob/main/CV_%D1%83%D1%80%D0%BE%D0%BA_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import time
import subprocess
import threading
import textwrap
import sys

# ===== 1. Очистка предыдущих процессов =====
print("Очистка старых процессов...")
subprocess.run("pkill -9 -f cloudflared 2>/dev/null || true", shell=True)
subprocess.run("pkill -9 -f server.py 2>/dev/null || true", shell=True)
subprocess.run("fuser -k 5000/tcp 2>/dev/null || true", shell=True)  # более надёжно, чем lsof

# ===== 2. Установка зависимостей =====
print("Установка Python пакетов...")
subprocess.run("pip install flask flask-cors ultralytics --quiet", shell=True, check=True)

# Установка cloudflared
print("Установка cloudflared...")
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb", shell=True, check=True)
subprocess.run("dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1", shell=True)
subprocess.run("rm cloudflared-linux-amd64.deb", shell=True)

# Проверка ffmpeg (в Colab обычно есть, но на всякий случай)
try:
    subprocess.run("ffmpeg -version", shell=True, capture_output=True, check=True)
except:
    print("ffmpeg не найден, устанавливаю...")
    subprocess.run("apt-get update -qq && apt-get install -y ffmpeg", shell=True, check=True)

# ===== 3. Создание конфига трекера =====
tracker_config = """\
tracker_type: bytetrack
track_high_thresh: 0.4
track_low_thresh: 0.1
new_track_thresh: 0.7
track_buffer: 300
match_thresh: 0.95
fuse_score: true
"""
with open("/content/custom_bytetrack.yaml", "w") as f:
    f.write(tracker_config)
print("Конфиг трекера создан.")

# ===== 4. Генерация server.py (с улучшенным логированием) =====
server_code = '''import os
import uuid
import cv2
import base64
import subprocess
import threading
import re
import time
from collections import Counter
from flask import Flask, request, jsonify, Response, send_file
from flask_cors import CORS
from ultralytics import YOLO

app = Flask(__name__)
CORS(app)

UPLOAD_FOLDER = "/content/uploads"
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
TARGET_CLASSES = [0, 1, 2, 3, 5, 7]  # person, bicycle, car, motorcycle, bus, truck
TRACKER_PATH = "/content/custom_bytetrack.yaml"
MODEL_PATH = "yolo11m.pt"

print("[SERVER] Загрузка модели YOLO...")
model = YOLO(MODEL_PATH)
print("[SERVER] Модель загружена.")

tasks = {}

def process_video(task_id, input_path, confidence, output_raw, output_final):
    """Фоновая обработка видео."""
    cap = cv2.VideoCapture(input_path)
    total_frames = max(1, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_raw, fourcc, fps, (width, height))

    class_votes = {}
    unique_ids = {}
    frame_idx = 0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            results = model.track(
                frame,
                persist=True,
                tracker=TRACKER_PATH,
                conf=confidence,
                classes=TARGET_CLASSES,
                verbose=False
            )

            plotted = results[0].plot()
            out.write(plotted)

            boxes = results[0].boxes
            if boxes is not None and boxes.id is not None:
                track_ids = boxes.id.cpu().numpy().astype(int)
                class_ids = boxes.cls.cpu().numpy().astype(int)
                for tid, cls_id in zip(track_ids, class_ids):
                    class_name = model.names[cls_id]
                    if tid not in class_votes:
                        class_votes[tid] = Counter()
                    class_votes[tid][class_name] += 1
                    stable_name = class_votes[tid].most_common(1)[0][0]
                    unique_ids.setdefault(stable_name, set()).add(tid)

            if frame_idx % 5 == 0:
                progress = int((frame_idx + 1) * 100 / total_frames)
                live_stats = {name: len(ids) for name, ids in unique_ids.items()}

                preview = plotted.copy()
                h, w = preview.shape[:2]
                new_w = 800
                new_h = int(h * new_w / w)
                preview = cv2.resize(preview, (new_w, new_h))
                _, buffer = cv2.imencode('.jpg', preview, [cv2.IMWRITE_JPEG_QUALITY, 70])
                preview_b64 = base64.b64encode(buffer).decode('ascii')

                with tasks[task_id]["lock"]:
                    tasks[task_id]["progress"] = progress
                    tasks[task_id]["preview_base64"] = preview_b64
                    tasks[task_id]["live_stats"] = live_stats

            frame_idx += 1

        cap.release()
        out.release()

        # Конвертация в MP4
        ffmpeg_cmd = [
            "ffmpeg", "-y", "-i", output_raw,
            "-vf", "scale=1280:-2",
            "-c:v", "libx264", "-preset", "fast",
            "-crf", "23", "-pix_fmt", "yuv420p",
            "-an", output_final
        ]
        subprocess.run(ffmpeg_cmd, check=True, capture_output=True)

        final_unique = {name: len(ids) for name, ids in unique_ids.items()}
        result_data = {
            "total_frames": total_frames,
            "unique_objects": final_unique
        }

        with tasks[task_id]["lock"]:
            tasks[task_id]["status"] = "completed"
            tasks[task_id]["progress"] = 100
            tasks[task_id]["live_stats"] = final_unique
            tasks[task_id]["result"] = result_data
            tasks[task_id]["output_video"] = output_final

        os.remove(output_raw)
        print(f"[SERVER] Задача {task_id} завершена.")

    except Exception as e:
        print(f"[SERVER] Ошибка в задаче {task_id}: {str(e)}")
        with tasks[task_id]["lock"]:
            tasks[task_id]["status"] = "error"
            tasks[task_id]["error"] = str(e)
        if os.path.exists(output_raw):
            os.remove(output_raw)
        if os.path.exists(output_final):
            os.remove(output_final)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok"})

@app.route('/upload', methods=['POST'])
def upload():
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400
    file = request.files['file']
    if file.filename == '':
        return jsonify({"error": "No selected file"}), 400

    try:
        confidence = float(request.form.get('confidence', 0.5))
    except ValueError:
        confidence = 0.5

    task_id = str(uuid.uuid4())
    input_path = os.path.join(UPLOAD_FOLDER, f"{task_id}_input.mp4")
    file.save(input_path)

    output_raw = os.path.join(UPLOAD_FOLDER, f"{task_id}_raw.avi")
    output_final = os.path.join(UPLOAD_FOLDER, f"{task_id}.mp4")

    tasks[task_id] = {
        "status": "processing",
        "progress": 0,
        "preview_base64": None,
        "live_stats": {},
        "result": None,
        "error": None,
        "output_video": None,
        "lock": threading.Lock()   # Lock создаётся сразу
    }

    thread = threading.Thread(
        target=process_video,
        args=(task_id, input_path, confidence, output_raw, output_final),
        daemon=True
    )
    thread.start()
    print(f"[SERVER] Задача {task_id} запущена.")

    return jsonify({"task_id": task_id})

@app.route('/status/<task_id>', methods=['GET'])
def status(task_id):
    task = tasks.get(task_id)
    if task is None:
        return jsonify({"status": "error", "error": "Task not found"}), 404

    with task["lock"]:
        if task["status"] == "error":
            return jsonify({"status": "error", "error": task["error"]})
        elif task["status"] == "completed":
            return jsonify({
                "status": "completed",
                "progress": 100,
                "live_stats": task["live_stats"],
                "result": task["result"]
            })
        else:
            return jsonify({
                "status": "processing",
                "progress": task["progress"],
                "preview_base64": task["preview_base64"],
                "live_stats": task["live_stats"]
            })

@app.route('/stream/<task_id>', methods=['GET'])
def stream_video(task_id):
    task = tasks.get(task_id)
    if task is None or task["status"] != "completed" or task["output_video"] is None:
        return "Video not found", 404

    video_path = task["output_video"]
    if not os.path.exists(video_path):
        return "Video file missing", 404

    size = os.path.getsize(video_path)
    range_header = request.headers.get('Range', None)

    def read_chunk(start, end):
        with open(video_path, 'rb') as f:
            f.seek(start)
            return f.read(end - start + 1)

    if range_header:
        match = re.search(r'bytes=(\d+)-(\d*)', range_header)
        if match:
            start = int(match.group(1))
            end_str = match.group(2)
            end = int(end_str) if end_str else size - 1
            if start >= size or end >= size or start > end:
                return Response("Invalid Range", status=416)
            data = read_chunk(start, end)
            response = Response(data, status=206, mimetype='video/mp4')
            response.headers.add('Content-Range', f'bytes {start}-{end}/{size}')
            response.headers.add('Accept-Ranges', 'bytes')
            response.headers.add('Content-Length', str(len(data)))
            return response

    data = read_chunk(0, size-1)
    response = Response(data, status=200, mimetype='video/mp4')
    response.headers.add('Accept-Ranges', 'bytes')
    response.headers.add('Content-Length', str(size))
    return response

@app.route('/download/<task_id>', methods=['GET'])
def download_video(task_id):
    task = tasks.get(task_id)
    if task is None or task["status"] != "completed" or task["output_video"] is None:
        return "Video not found", 404
    video_path = task["output_video"]
    if not os.path.exists(video_path):
        return "Video file missing", 404
    return send_file(video_path, as_attachment=True, download_name=f"{task_id}.mp4")

if __name__ == '__main__':
    print("[SERVER] Запуск Flask на порту 5000...")
    app.run(host='0.0.0.0', port=5000, threaded=True)
'''

with open("/content/server.py", "w") as f:
    f.write(server_code)
print("server.py записан.")

# ===== 5. Запуск сервера =====
print("Запуск сервера...")
srv = subprocess.Popen(
    ["python", "/content/server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# ===== 6. Фоновый поток для вывода логов сервера =====
def log_server():
    for line in srv.stdout:
        print(f"[SERVER] {line.strip()}")

threading.Thread(target=log_server, daemon=True).start()

# ===== 7. Ожидание готовности сервера (health check) =====
print("Ожидание запуска сервера (health check)...")
for i in range(30):  # максимум 30 секунд
    time.sleep(1)
    try:
        import requests
        resp = requests.get("http://localhost:5000/health", timeout=1)
        if resp.status_code == 200 and resp.json().get("status") == "ok":
            print("Сервер готов.")
            break
    except:
        pass
else:
    print("Сервер не ответил за 30 секунд. Проверьте логи выше.")
    # Не завершаем, возможно, ещё запустится

# ===== 8. Запуск Cloudflare туннеля =====
print("Запуск cloudflared...")
tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# ===== 9. Поиск URL туннеля =====
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
public_url = None

def log_tunnel():
    global public_url
    for line in tun.stdout:
        print(f"[TUNNEL] {line.strip()}")
        if not public_url:
            match = url_pattern.search(line)
            if match:
                public_url = match.group(0)
                print("\\n" + "="*60)
                print(f"✅ Публичный URL: {public_url}")
                print("📌 Эндпоинты:")
                print(f"   POST {public_url}/upload")
                print(f"   GET  {public_url}/status/<task_id>")
                print(f"   GET  {public_url}/stream/<task_id>")
                print(f"   GET  {public_url}/download/<task_id>")
                print(f"   GET  {public_url}/health")
                print("="*60 + "\\n")

threading.Thread(target=log_tunnel, daemon=True).start()

# ===== 10. Держим ячейку активной =====
print("Сервер и туннель запущены. Нажмите Ctrl+C для остановки.")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Завершение...")
    srv.terminate()
    tun.terminate()
    subprocess.run("pkill -9 -f cloudflared", shell=True)
    subprocess.run("pkill -9 -f server.py", shell=True)
    print("Остановлено.")

<>:266: SyntaxWarning: invalid escape sequence '\d'
<>:266: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_4063/1916610740.py:266: SyntaxWarning: invalid escape sequence '\d'
  match = re.search(r'bytes=(\d+)-(\d*)', range_header)


Очистка старых процессов...
Установка Python пакетов...
Установка cloudflared...
Конфиг трекера создан.
server.py записан.
Запуск сервера...
Ожидание запуска сервера (health check)...
[SERVER] [SERVER] Загрузка модели YOLO...
[SERVER] [SERVER] Модель загружена.
[SERVER] [SERVER] Запуск Flask на порту 5000...
[SERVER] * Serving Flask app 'server'
[SERVER] * Debug mode: off
[SERVER] WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
[SERVER] * Running on all addresses (0.0.0.0)
[SERVER] * Running on http://127.0.0.1:5000
[SERVER] * Running on http://172.28.0.12:5000
[SERVER] Press CTRL+C to quit
[SERVER] 127.0.0.1 - - [15/Mar/2026 20:46:55] "GET /health HTTP/1.1" 200 -
Сервер готов.
Запуск cloudflared...
Сервер и туннель запущены. Нажмите Ctrl+C для остановки.
[TUNNEL] 2026-03-15T20:46:56Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. Howe